In [0]:
%pip install vaderSentiment

In [0]:
import pyspark.sql.functions as F

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, avg, count
from pyspark.sql.types import FloatType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

df = spark.table("samples.bakehouse.media_customer_reviews")

analyzer = SentimentIntensityAnalyzer()


def analyze_sentiment_vader(text):
    if text is None:
        return 0.0
    # VADER returns a dictionary. 'compound' is the overall -1.0 to 1.0 score.
    sentiment_dict = analyzer.polarity_scores(text)
    return sentiment_dict["compound"]


sentiment_udf = udf(analyze_sentiment_vader, FloatType())

scored_df = df.withColumn("vader_compound", sentiment_udf(col("review"))).withColumn(
    "rating_10_scale", F.round((col("vader_compound") + 1.0) * 5, 1)
)

benchmark_df = (
    scored_df.groupBy("franchiseID")
    .agg(
        F.round(avg("rating_10_scale"), 2).alias("avg_rating_out_of_10"),
        count("new_id").alias("total_reviews"),
    )
    .orderBy(col("avg_rating_out_of_10").desc())
)

benchmark_df.write.format("delta").mode("overwrite").saveAsTable(
    "ctl_training_dev.m3.yo_bakehouse_franchise_rating"
)